In [ ]:
x = 0
lr = 0.05
T = 3
b = 0.9
v = 0

for epoch in range(T):
    dy = 2*x - 6
    x = x - lr*dy
    print(round(x, 2))


print()
x = 0
lr = 0.05
T = 3
b = 0.9
v = 0

for epoch in range(T):
    dy = 2*x - 6
    v = b*v + dy
    x = x - lr*v
    print(f'{round(x, 2)} : {v}')


0.3
0.57
0.81

0.3 : -6.0
0.84 : -10.8
1.54 : -14.040000000000001


In [ ]:
for epoch in range(2):
    gt = 2*x - lr*b*v - 3
    v = b*v + gt
    x = x - lr*v
    print(f'{round(x, 2)} : {v}')

2.61 : -4.064233605
2.67 : -1.2502692717750001


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class NestedLearningLayer(nn.Module):
    def __init__(self, input_dim, memory_dim, learning_rate=0.1):
        super().__init__()
        # Level 1 Parameters (Static - Updated during global training)
        self.w_key = nn.Linear(input_dim, memory_dim)
        self.w_value = nn.Linear(input_dim, memory_dim)
        self.w_query = nn.Linear(input_dim, memory_dim)

        # Hyperparameters for Level 2 (The internal optimizer)
        self.lr = learning_rate
        self.memory_dim = memory_dim

    def forward(self, x):
        """
        x: Input sequence of shape (Batch, Seq_Len, Input_Dim)
        """
        batch_size, seq_len, _ = x.shape

        # 1. Project inputs to Keys, Values, and Queries (Level 1 computation)
        k = self.w_key(x)   # The "input" to the internal memory
        v = self.w_value(x) # The "target" for the internal memory
        q = self.w_query(x) # The "test query" for the internal memory

        # Initialize Memory M_0 (Level 2 State)
        # In a real model, this could be meta-learned. Here, it's zero.
        M = torch.zeros(batch_size, self.memory_dim, self.memory_dim, device=x.device)

        outputs = []

        # 2. Iterate through the sequence (This is the Level 2 Optimization Loop)
        for t in range(seq_len):
            # Current token's data
            k_t = k[:, t, :].unsqueeze(2) # Shape: (B, D, 1)
            v_t = v[:, t, :].unsqueeze(2) # Shape: (B, D, 1)
            q_t = q[:, t, :].unsqueeze(2) # Shape: (B, D, 1)

            # --- The Nested Learning Step ---

            # A. PREDICTION (Inference using current Memory state)
            # The output depends on the memory learned from PAST tokens (0 to t-1)
            y_t = torch.bmm(M, q_t).squeeze(2)
            outputs.append(y_t)

            # B. OPTIMIZATION (Training the Memory on current token)
            #? We want M to map k_t to v_t.
            # Objective: L = || M * k_t - v_t ||^2 (L2 Regression)
            # Gradient of L w.r.t M: (M * k_t - v_t) * k_t^T

            # Compute error (prediction vs target value)
            prediction = torch.bmm(M, k_t)
            error = prediction - v_t

            # Compute Gradient
            gradient = torch.bmm(error, k_t.transpose(1, 2))

            # Update Memory (Gradient Descent Step)
            # M_{t+1} = M_t - lr * Gradient
            M = M - self.lr * gradient

        return torch.stack(outputs, dim=1)

# Example Usage
# Input: Batch of 2, Sequence Length 5, Dimension 4
input_data = torch.randn(2, 5, 4)
model = NestedLearningLayer(input_dim=4, memory_dim=4)
output = model(input_data)

print("Input shape:", input_data.shape)
print("Output shape:", output.shape)
print("\nThe layer learned from tokens 0-3 to predict token 4!")

OSError: [WinError 126] The specified module could not be found. Error loading "d:\Utilities\miniconda\envs\continual_clip\lib\site-packages\torch\lib\caffe2_nvrtc.dll" or one of its dependencies.